In [1]:
import numpy as np
import pandas as pd
from bokeh.io import output_notebook, show
from bokeh.models import ColumnDataSource
from bokeh.plotting import figure
import os

os.environ["BOKEH_MINIFIED"] = "false"
output_notebook()

Loading BokehJS ...

# Bokeh 与 JavaScript

Bokeh 的绘制与交互功能由跑在浏览器里的 JavaScript 库负责，用 Python 做图时加上 JavaScript 的代码，就能增加交互功能，这一章说明它的写法。

为了便于在浏览器里调试，在读入 Bokeh 之前把环境变量 `BOKEH_MINIFIED` 设成 `"false"`，这样 `output_notebook()` 会读没有压缩的版本，代码也好跟。

```python
import os
from bokeh.io import output_notebook

os.environ["BOKEH_MINIFIED"] = "false"
output_notebook()
```

在 Notebook 里执行下面的 JavaScript，输出里出现 `ColumnDataSource` 就说明读的是没压缩的版本；`Bokeh.Models(名字)` 能取到那个名字的模型的类。

这一章按下面的顺序说明。

- 在浏览器里调试的办法
- 事件的回调(`js_on_event()`)
- 数据的变换器(`dodge()`、`jitter()`、颜色映射)
- 筛选数据源(过滤器)
- 刻度的文字、交互工具

In [2]:
%%javascript
let ColumnDataSource = Bokeh.Models["ColumnDataSource"];
element.append(ColumnDataSource.name)

<IPython.core.display.Javascript object>

## 调试 JavaScript

先看在浏览器里调试 JavaScript 的办法。为了能从 JavaScript 方便地取到图里的对象，做图时用 `id` 给图一个 ID，做图形时用 `name` 给名字。

In [3]:
from bokeh.models import ColumnDataSource

source = ColumnDataSource(data=dict(x=[1, 2, 3], y=[1, 3, 2]))
fig = figure(height=250)
fig._id = "my_fig"
line = fig.line("x", "y", source=source, name="my_line")
show(fig)

在 Notebook 里执行上面的程序之后，在浏览器里按 [Ctrl+Shift+I] 打开开发者工具，在 Console 的标签页执行下面的 JavaScript，就会输出折线的 x 与 y 的数据。这些变量是全局的，所以从 Console 里能看 `fig`、`line`、`source` 等的内容。

```javascript
fig = Bokeh.index["my_fig"].model; //❶
line = fig.select_one("my_line");  //❷
source = line.data_source;         //❸
console.log(source.data);
```

❶ 在 JavaScript 里，图对应 `Bokeh.index` 里的 `PlotView`，它的 `model` 就等于 Python 的 `fig`，❷ 用 `select_one()` 取到 `name` 为 `"my_line"` 的模型，对应 Python 的 `line`(`GlyphRenderer`)，❸ 它的 `data_source` 对应 Python 的 `source`(`ColumnDataSource`)。

也就是说，Bokeh 在 JavaScript 里也做出与 Python 结构相同的对象。

![](images/bokeh-javascript-debug.png)

## `js_on_event()` 的回调

`Model` 有 `js_on_change()` 与 `js_on_event()`：前者在属性变化时、后者在交互事件发生时调用 JavaScript。回调里可以从 `cb_obj` 取到事件的信息，事件的类在 `bokeh.events` 里，`cb_obj` 的属性与那个事件的属性相同。

例如看双击的 `DoubleTap`，`cb_obj` 里有 `x`、`y`、`sx`、`sy` 4 个：`x`、`y` 是数据的坐标，`sx`、`sy` 是画面的像素。

In [4]:
from bokeh import events

print(events.DoubleTap.__doc__)

Announce a double-tap or double-click event on a Bokeh plot.

Attributes:
    sx (float) : x-coordinate of the event in *screen* space
    sy (float) : y-coordinate of the event in *screen* space
    x (float) : x-coordinate of the event in *data* space
    y (float) : y-coordinate of the event in *data* space




下面的例子能用鼠标画曲线：在图上双击开始画，再双击一次就停止，期间记录鼠标的轨迹。`DoubleTap` 与 `MouseMove` 的回调往数据源里加点。

❶ `source` 是放轨迹的数据源，在它的 `tags` 里放颜色的列表与「现在是否在画」；好几个图形都连在这个数据源上。❷ `point_source` 是放鼠标当前位置的数据源，散点图连在它上面，❸ 两个事件的回调都是 `callback`。❹ 双击时 `event_name` 是 `'doubletap'`，所以切换要不要画，❺ 画的时候把鼠标的位置加进轨迹。

In [5]:
from bokeh.models import CustomJS
from bokeh.events import MouseMove, DoubleTap
from bokeh.palettes import Category20_20

fig = figure(
    active_drag=None,
    x_range=(-10, 10),
    y_range=(-10, 10),
    frame_width=500,
    frame_height=300,
)
source = ColumnDataSource(
    data=dict(xs=[], ys=[], colors=[]), tags=[dict(colors=Category20_20, mode="normal")]
)  # ❶
pointer_source = ColumnDataSource(data=dict(x=[0], y=[0]))  # ❷

lines = fig.multi_line(
    "xs", "ys", line_color="colors", source=source, line_width=2
)  # ❶
pointer = fig.scatter(
    "x", "y", source=pointer_source, visible=False, marker="cross", size=10, color="red"
)  # ❷

callback = CustomJS(  # ❸
    args=dict(source=source, pointer=pointer),
    code="""
let data = source.data;
let info = source.tags[0];

if(cb_obj.event_name == 'doubletap'){
    if(info.mode == 'normal'){ //❹
        let colors = info.colors;
        data.xs.push([]);
        data.ys.push([]);
        data.colors.push(colors[data.xs.length % colors.length]);
        pointer.visible = true;
        pointer.data_source.change.emit();
        info.mode = 'draw';
    }
    else{
        info.mode = 'normal';
        pointer.visible = false;
    }
}
else if(cb_obj.event_name == 'mousemove'){ //❺
    pointer.data_source.data["x"] = [cb_obj.x];
    pointer.data_source.data["y"] = [cb_obj.y];
    if(info.mode == 'draw'){
        let xs = data.xs;
        let ys = data.ys;
        xs[xs.length-1].push(cb_obj.x);
        ys[ys.length-1].push(cb_obj.y);
        source.change.emit();
        pointer.data_source.change.emit();
    }
}
""",
)

fig.js_on_event(DoubleTap, callback)
fig.js_on_event(MouseMove, callback)
show(fig)

## 数据的变换器

数据常常不能直接拿来显示，那时可以在 Python 里转换，也可以用 JavaScript 的变换器(`Transform`)：把数值变成颜色、把点错开来避免重叠，这些都能在 JavaScript 里做。

下面例子的 `dodge("y", 1)` 把 `source` 的 `"y"` 列加上 1，当作红色曲线的 y。

In [6]:
from bokeh.transform import dodge

x = np.linspace(0, 2 * np.pi, 100)
y = np.sin(x)
source = ColumnDataSource(data=dict(x=x, y=y))
fig = figure(frame_width=400, frame_height=250)
fig.line("x", "y", source=source)
line = fig.line("x", dodge("y", 1), color="red", source=source)
show(fig)

用变换器时，图形的 `y` 不是列名的字符串，而是表示变换的字典，意思是给 `source` 的 `y` 列套上 `Dodge`；`Dodge` 是变换器的模型，`dodge()` 是做出它的函数。

In [7]:
line.glyph.y

Field(field='y', transform=Dodge(id='p1153', ...), units=Unspecified)

点的坐标完全相同时，散点图看不出分布，所以用 `jitter()` 给坐标加一点扰动。下面的程序用 `np.round()` 把随机数取整，所以点排在格子上(左图)；用 `jitter()` 给 x 与 y 加上扰动，分布就看出来了(右图)。

In [8]:
from bokeh.transform import jitter
from bokeh.layouts import row

x, y = np.round(np.random.normal(scale=0.2, size=(2, 500)), 1)

source = ColumnDataSource(data=dict(x=x, y=y))
fig1 = figure(frame_width=250, frame_height=250)
fig1.scatter("x", "y", source=source, alpha=0.3)
fig2 = figure(frame_width=250, frame_height=250)
fig2.scatter(
    jitter("x", 0.1, distribution="normal"),
    jitter("y", 0.1, distribution="normal"),
    source=source,
    alpha=0.3,
)
layout = row(fig1, fig2)
show(layout)

下面的例子用数据源的值决定点的颜色。

❶ `linear_cmap()` 做出把数值变成颜色的变换器，除了列名之外的 3 个参数是颜色的列表、最小值与最大值：它把 `z` 列的值对应到 `Viridis` 的颜色。❷ 把变换器放进点的 `fill_color`，❸ 用 `ColorBar` 能把值与颜色的对应显示出来，它的 `color_mapper` 就是变换用的 `ColorMapper`。

In [9]:
from bokeh.transform import linear_cmap
from bokeh.palettes import Viridis
from bokeh.models import ColorBar

x, y = np.random.normal(scale=0.2, size=(2, 500))
z = (x**2 + y**2) ** 0.5

source = ColumnDataSource(data=dict(x=x, y=y, z=z))
fig_cmap1 = figure(frame_width=400, frame_height=300)
cmap = linear_cmap("z", Viridis[256], z.min(), z.max())  # ❶
c = fig_cmap1.scatter(
    "x", "y", fill_color=cmap, line_color=None, source=source, alpha=1, size=6
)  # ❷
colorbar = ColorBar(
    color_mapper=cmap["transform"],  # ❸
    label_standoff=12,
    border_line_color=None,
    location=(0, 0),
)
fig_cmap1.add_layout(colorbar, "right")

`linear_cmap()` 把连续的值变成连续的颜色，而 `factor_cmap()` 把分类的值变成颜色：除了列名之外的 2 个参数是色图与分类的列表，按分类的顺序取色。

❶ 读 `iris.csv` 时把 `species` 列设成分类型。❷ 分类型的列用 `cat.categories` 能取到所有分类；这样一来，一张散点图就能显示出各分类的分布。

In [10]:
from bokeh.transform import factor_cmap
from bokeh.palettes import Category10

df = pd.read_csv("data/iris.csv", dtype={"species": "category"})  # ❶

source = ColumnDataSource(data=df)
fig_cmap2 = figure(frame_width=400, frame_height=300)
cmap = factor_cmap("species", Category10[10], df.species.cat.categories)  # ❷
c = fig_cmap2.scatter(
    "sepal_length",
    "sepal_width",
    fill_color=cmap,
    legend_field="species",
    line_color=None,
    source=source,
    alpha=1,
    size=6,
)

到这里的变换器的 JavaScript 都包含在 Bokeh 里，不必自己写；要自己写就用 `CustomJSTransform`：`func` 变换单个值、`v_func` 变换数组，`func` 里 `x` 是要变换的值、返回值是结果，`v_func` 里 `xs` 是数组、返回值是变换后的数组。

下面的例子 ❶ 做出 `CustomJSTransform`，计算 $f(x) = \sin(x) \cdot \sin(10 x)$；因为要把整列数据一起变换，所以不写 `func`，❷ 用 `transform()` 给 `x` 列套上这个变换器。

In [11]:
layout_cmap = row(fig_cmap1, fig_cmap2)
show(layout_cmap)

In [12]:
from bokeh.transform import transform
from bokeh.models import CustomJSTransform

x = np.linspace(0, 2 * np.pi, 1000)

source = ColumnDataSource(data=dict(x=x))
fig_cjs1 = figure(frame_width=400, frame_height=250)

transform_sin = CustomJSTransform(  # ❶
    func="return Math.sin(x) * Math.sin(10*x);",
    v_func="return xs.map(x=>Math.sin(x) * Math.sin(10*x));",
)

fig_cjs1.line("x", transform("x", transform_sin), source=source);  # ❷

`v_func` 里的 `xs` 只有那一列的数据；把别的模型放进 `args`，`v_func` 里也能用。下面的例子把 `source` 放进 `args`，返回 `x` 列与 `y` 列平方和的平方根，当作点的大小。

In [13]:
x, y = np.random.normal(scale=0.2, size=(2, 500))
source = ColumnDataSource(data=dict(x=x, y=y))
fig_cjs2 = figure(frame_width=400, frame_height=300)

transform_dist = CustomJSTransform(
    args=dict(source=source),
    v_func="""
        let data = source.data;
        let ys = data.y;
        return xs.map((x, i) => {
            let y = ys[i];
            return 10 * Math.sqrt(x*x + y*y);
        })
    """,
)

c = fig_cjs2.scatter(
    "x",
    "y",
    size=transform("x", transform_dist),
    line_color=None,
    source=source,
    alpha=1,
)

In [14]:
layout_cjs = row(fig_cjs1, fig_cjs2)
show(layout_cjs)

## 筛选数据源

在 JavaScript 里也能筛选数据源的数据，下面的例子里用下拉框选一个分类，就只显示那个分类的点。

❶ 在下拉框的 `options` 里放 `species` 列的值，先选中第一个。❷ `GroupFilter` 只保留某一列等于某个值的数据，`column_name` 是列名、`group` 是那个值。❸ 把过滤器放进 `CDSView` 的 `filters`，`source` 里放要筛选的数据源，❹ 做图形时用 `view` 把这个视图传进去。❺ 下拉框的值一变，`callback` 就会跑，❻ 选中的值在 `cb_obj.value` 里，把它放进过滤器的 `group`，❼ 最后用 `change.emit()` 把图重画。

In [15]:
from bokeh.layouts import column
from bokeh.models import CDSView, GroupFilter, Select

df = pd.read_csv("data/iris.csv", dtype={"species": "category"})

categories = df.species.cat.categories.tolist()
select = Select(options=categories, value=categories[0])  # ❶
source = ColumnDataSource(data=df)
group_filter = GroupFilter(column_name="species", group=select.value)  # ❷
view = CDSView(filter=group_filter)  # ❸

fig_filter1 = figure(frame_width=400, frame_height=300)
c = fig_filter1.scatter(
    "sepal_length",
    "sepal_width",
    line_color=None,
    source=source,
    view=view,
    alpha=1,
    size=6,
)  # ❹

callback = CustomJS(
    args=dict(filter=group_filter, source=source),
    code="""
    filter.group = cb_obj.value; //❻
    source.change.emit();        //❼
    """,
)

select.js_on_change("value", callback)  # ❺
layout_filter1 = column(select, fig_filter1)

用 `CustomJSFilter` 就能用自己的代码筛选：`args` 与 `code` 的用法与 `CustomJS` 相同，返回值是整数或真假的数组，用法与 NumPy 的索引一样。

下面的例子 ❶ 用 `RangeSlider` 选 `petal_length` 的范围，❷ 为了在代码里读到滑块的值，把滑块放进 `args`。❸ 代码里 `source` 是要筛选的数据源。❹ 对列的每个值判断是否落在范围里：用 `Array.from(data)` 变成 JavaScript 的数组，再用 `map()` 做出真假的数组。

In [16]:
from bokeh.models import CustomJSFilter, RangeSlider

source = ColumnDataSource(data=df)
fig_filter2 = figure(frame_width=400, frame_height=300)
slider = RangeSlider(  # ❶
    start=df.petal_length.min(),
    end=df.petal_length.max(),
    value=(df.petal_length.min(), df.petal_length.max()),
    step=0.1,
    title="petal length",
)

func_filter = CustomJSFilter(  # ❷
    args=dict(slider=slider),
    code="""
    let data = source.data['petal_length']; //❸
    let [start, end] = slider.value;
    let res = Array.from(data).map(v => start <= v && v <= end); //❹
    return res;
    """,
)

view = CDSView(filter=func_filter)

c = fig_filter2.scatter(
    "sepal_length",
    "sepal_width",
    fill_color=linear_cmap(
        "petal_length", Viridis[256], df.petal_length.min(), df.petal_length.max()
    ),
    line_color=None,
    source=source,
    view=view,
    alpha=1,
    size=6,
)

callback = CustomJS(args=dict(source=source), code="source.change.emit();")

slider.js_on_change("value", callback)
layout_filter2 = column(slider, fig_filter2)

In [17]:
layout_filter = row(layout_filter1, layout_filter2)
show(layout_filter)

## 改变刻度的文字

`FuncTickFormatter` 用 JavaScript 做出刻度的文字。下面的例子比较几种说法在概率上的分布：y 轴的刻度是那些说法，曲线是它们分布的估计。

❶ 从 Bokeh 的示例数据里读 `probly`，每一列是对一种说法的概率评价。❷ 用 `gaussian_kde()` 估计每一列的分布；为了方便比较，把曲线的最大值都归一成 2。❸ 把估计的结果收进一个 `DataFrame`，用 `zscore()` 按列标准化。

In [18]:
from bokeh.sampledata.perceptions import probly  # ❶
from scipy.stats import gaussian_kde
import numpy as np

x = np.linspace(0, 100, 200)
df_kde = probly.apply(lambda s: gaussian_kde(s)(x)).set_index(x)  # ❷
df_kde = df_kde * (2 / df_kde.max())  # ❸

❹ 在循环里逐列画曲线，y 的值用 `dodge()` 错开，❺ 用 `FixedTicker` 决定 y 的刻度位置。❻ 用 `FuncTickFormatter` 做出刻度的文字：`tick` 是刻度的位置，用它从 `labels` 里取文字。

In [19]:
from bokeh.models import FixedTicker, CustomJSTickFormatter
from bokeh.palettes import Category20_20

source = ColumnDataSource(df_kde)
fig_tick = figure()

for i, col in enumerate(df_kde.columns):
    fig_tick.line(
        "index",
        dodge(col, i),
        source=source,
        color=Category20_20[i],
        line_width=2,
        alpha=0.6,
    )  # ❹

fig_tick.yaxis.ticker = FixedTicker(ticks=np.arange(df_kde.shape[1]))  # ❺
fig_tick.yaxis.ticker.tags = ["x", "y", "z"]
fig_tick.yaxis.formatter = CustomJSTickFormatter(
    args=dict(labels=list(df_kde.columns)), code="return labels[tick];"
)  # ❻
show(fig_tick)

## 交互工具

工具栏里有 3 个能调用 JavaScript 的工具：`CustomAction` 在按下按钮时动、`TapTool` 在点击图形时动、`HoverTool` 在鼠标移到图形上时动。

下面的例子用 `HoverTool` 显示中国各个省的说明：鼠标放到省上，右边的栏里就出现说明。

❶ `china_simple.json` 是各省形状的 GeoJSON，`GeoJSONDataSource` 能直接读，❷ `china_info.json` 是各省的说明，用 `json.load()` 变成字典，❸ 用 `patches()` 画出各省的形状，颜色由 `population` 列决定。❹ 用 `Div` 显示说明，在它的 `tags` 里放各省说明的字典(Python 的字典会变成 JavaScript 的对象)，❺ 在 `HoverTool` 的 `renderers` 里放要响应的图形。

❻ 回调里 `cb_data.index.indices` 是鼠标下方图形的编号，不是空的话就用第一个。❼ 从 `name` 列取到省名，❽ 再从说明的字典里取出文字，放进 `div_info.text`。

In [20]:
import json
from bokeh.models import GeoJSONDataSource, Div, CustomJS, HoverTool
from bokeh.layouts import row
from bokeh.transform import linear_cmap
from bokeh.palettes import Viridis

with open("data/china_simple.json", encoding="utf-8") as f:  # ❶
    geojson_string = f.read()

with open("data/china_info.json", encoding="utf-8") as f:  # ❷
    province_info = json.load(f)

source = GeoJSONDataSource(geojson=geojson_string)
fig = figure(match_aspect=True, aspect_scale=1, frame_width=500, frame_height=500)
patches = fig.patches(
    "xs",
    "ys",
    source=source,
    line_color="black",  # ❸
    fill_color=linear_cmap("population", Viridis[256], low=None, high=None),
)

div_info = Div(width=300, tags=[province_info])  # ❹

callback = CustomJS(
    args=dict(source=source, div_info=div_info),
    code="""
    let indices = cb_data.index.indices; //❻
    let info = div_info.tags[0];

    if(indices.length > 0){
        let name = source.data['name'][indices[0]]; //❼
        let text = info[name]; //❽
        div_info.text = `<h2>${name}</h2><p>${text}</p>`; 
    }
    """,
)

hover = HoverTool(tooltips=None, callback=callback, renderers=[patches])  # ❺
fig.tools.append(hover)
layout = row(fig, div_info)
show(layout)

## 小结

* Bokeh 的图与交互由浏览器的 JavaScript 驱动，JavaScript 那边也有与 Python 模型结构相同的对象。
* 给图加上 `id` 与 `name` 之后，就能在开发者工具的 Console 里通过 `Bokeh.index` 访问。
* 属性的变化用 `js_on_change()`、事件用 `js_on_event()` 来接，信息在 `cb_obj` 里。
* 变换器 Bokeh 里已经备好(`dodge()`、`jitter()`、`linear_cmap()`、`factor_cmap()`)，要自己写就用 `CustomJSTransform`。
* 筛选用 `CDSView` 的 `filters`(`GroupFilter`、`CustomJSFilter`)来做，并把 `view` 传给图形。
* 刻度的文字用 `FuncTickFormatter`、位置用 `FixedTicker` 决定。
* 工具 `CustomAction`、`TapTool`、`HoverTool` 都能挂上 JavaScript。